
# 5. Sustainability Analysis of Quiet Outperformers

This analysis addresses the research question 4 by comparing the stability/volatility of "Outperformer" channels against a carefully constructed control group of non-outperforming channels, using **Propensity Score Matching (PSM)** to mitigate selection bias inherent in observational data. Due to the complexity of this analysis, it is divided into two Jupyter notebooks mentioned below, that can be consulted for full details. In the next milestone, we will present the complete results and findings.


### I. Initial Feasibility and Metric Definition (`tests/sustainability_test_channel.ipynb`)

Initial data processing and feasibility assessment, ensuring that the required time-series data was present and reliable for metric calculation.

* **Data Handling and Filtering:** We began by merging the channel metadata with the **weekly time-series data**. We established the **analysis window** for stability measurement as **12 months (52 weeks)**. This specific subset was chosen for computational feasibility, allowing for robust conclusions to be drawn from the most recent yearly data. This confirmed that the data size was manageable and allowed us to isolate a high-quality subset for analysis.
* **Metric Definition:** To quantify stability, we defined and calculated the **Coefficient of Variation (CV)** for both weekly change in **Views** ($\text{CV}_{\Delta \text{views}}$) and weekly change in **Subscribers** ($\text{CV}_{\Delta \text{subs}}$).
    $$
    \text{CV} = \frac{\text{Standard Deviation } (\sigma \text{ of weekly growth})}{\text{Mean } (\mu \text{ of weekly growth})}
    $$
    This metric provides a robust, **scale-independent** measure of volatility. Because it is a ratio, it effectively normalizes the growth stability for channels of **vastly different sizes** (e.g., channels with 10k subscribers vs. 1M subscribers), where a **lower CV indicates higher sustainability/stability**. The feasibility of calculating this metric across thousands of channels was confirmed.


### II. Causal Analysis and Matching Methodology (`tests/sustainability_outperformers_vs_control.ipynb`)

The core of this notebook focused on addressing the fundamental challenge of selection bias to ensure a fair comparison.

* **Methodology (Propensity Score Matching - PSM):** To isolate the causal effect of "Outperforming" from its correlated covariates, we employed $1:1$ nearest-neighbor PSM.

* **Propensity Score Model:** We trained a Logistic Regression model to predict the probability of being an Outperformer using the following covariates:
    $$\text{is\_outperformer} \sim \text{log\_channel\_age\_days} + \text{log\_videos\_cc} + \text{log\_subscribers\_cc} + I\!\left(\text{log\_subscribers\_cc}^2\right)$$
    **Note on Feature Engineering**: The covariate **`channel_age_days`** was not present in the initial dataset. It was created during the preprocessing pipeline by calculating the difference in days between the data collection date and the channel's **`join_date`**. This feature is critical as channel longevity is a major confounder of performance.

    We used $\text{log()}$ transformation on the continuous features (`subscribers_cc`, `channel_age_days`, `videos_cc`) to address the **extreme skewness** in YouTube data. We also included a $\mathbf{I(\dots^2)}$ quadratic term for subscribers and applied $\mathbf{L2\ regularization}$ to stabilize the model coefficients against outliers.

* **Matching and Balance:** Matching was performed using an extremely tight $\mathbf{0.0005}$ caliper alongside **exact matching** on the `category_cc` variable to ensure that channels were only compared within the same content category (e.g., Gaming, Education).

    To verify the success of the matching process, we calculate the **Standardized Mean Difference (SMD)**:

    $$
    \text{SMD} = \frac{\bar{X}_{\text{Treated}} - \bar{X}_{\text{Control}}}{\sqrt{\frac{S_{\text{Treated}}^2 + S_{\text{Control}}^2}{2}}}
    $$

    where $\bar{X}$ is the mean of the covariate in each group, and $S^2$ is the variance.

    SMD is a crucial diagnostic: **Before matching**, a large SMD ($>0.1$) justifies the need for PSM. **After matching**, an SMD less than $0.1$ is the threshold used to confirm that the observed covariates are balanced, successfully minimizing selection bias. We iteratively refined the PSM model until the SMD for the continuous covariates was minimized.

    * Final SMD Check: `log_subscribers_cc` (SMD: **-0.0835**) and `log_channel_age_days` (SMD: **0.0357**) are now **well-balanced** ($|\text{SMD}| < 0.1$). `log_videos_cc` is slightly unbalanced (SMD: **-0.1187**).
    * **Conclusion:** Given the high balance achieved on the two most critical covariates (subscribers and channel age), this matching configuration is **acceptable** for proceeding to the outcome analysis.

* **Final Analysis:** The final analysis on the resulting balanced sample was conducted using the **Mann-Whitney U Test**. This non-parametric test was chosen because the CV stability metrics are **not normally distributed**, allowing us to reliably test the hypothesis that Outperformers and Control channels have different median growth volatility. In the initial results obtained, we observed that Outperformers tend to have a higher median CV in both views and subscribers growth compared to the Control group, suggesting that while they outperform, they may do so with greater volatility.